# 00 — Simulation Layer: Dataset Enrichment
**Purpose:** Derive missing metrics (Conversions, Spend reconciliation) and simulate e-commerce Average Order Value (AOV) & Revenue to enable ROI analysis.
**Inputs:** `data/raw/raw_dataset.csv` (200,000 rows × 20 columns)
**Outputs:** `data/raw/simulated_marketing.csv` (200,000 rows × 29 columns)

> ⚠️ **Simulation label:** `AOV` and `Revenue` columns are simulated. All original raw columns are preserved **unmodified**.

**Seed:** `np.random.seed(42)` is set at the top of every randomised block.


## 1. Imports & Seed
Standard analytical stack. Seed is set globally here and reset before any `np.random` call.


In [1]:
# Path Resolver Helper: Ensures notebook runs seamlessly from both root and notebooks/ directories
import os
from pathlib import Path

BASE_DIR = Path('..') if Path('../data').exists() else Path('.')
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_PROC = BASE_DIR / 'data' / 'processed'
DATA_SMPL = BASE_DIR / 'data' / 'sample'
IMAGE_DIR = BASE_DIR / 'images'
SQL_DIR = BASE_DIR / 'sql'
DASH_DIR = BASE_DIR / 'dashboard' / 'data'

for p in [DATA_RAW, DATA_PROC, DATA_SMPL, IMAGE_DIR, SQL_DIR, DASH_DIR]:
    p.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np

np.random.seed(42)
SEED = 42

print(f"pandas {pd.__version__} | numpy {np.__version__}")


pandas 3.0.5 | numpy 2.5.2


## 2. Load Raw Data
We load the original file and **never overwrite it**. All outputs go to a new enriched file.


In [2]:
df = pd.read_csv(str(DATA_RAW / 'raw_dataset.csv'))
print(f"Shape: {df.shape}")
print(f"Nulls:\n{df.isnull().sum().to_string()}")
df.head(3)


Shape: (200000, 16)
Nulls:
Campaign_ID         0
Company             0
Campaign_Type       0
Target_Audience     0
Duration            0
Channel_Used        0
Conversion_Rate     0
Acquisition_Cost    0
ROI                 0
Location            0
Language            0
Clicks              0
Impressions         0
Engagement_Score    0
Customer_Segment    0
Date                0


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,"$16,174.00",6.29,Chicago,Spanish,506,1922,6,Health & Wellness,01-01-2021
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,"$11,566.00",5.61,New York,German,116,7523,7,Fashionistas,02-01-2021
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,"$10,200.00",7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,03-01-2021


## 3. Parse Existing Columns (Type Corrections)
Two columns need type-conversion before arithmetic:
- **`Acquisition_Cost`** stored as `'$12,504.00'` → strip currency symbols → float
  - **Interpretation:** This is the **total campaign budget (Spend)** per row, not a per-conversion CPA.
    At mean ~$12,504 spend and mean ~44 conversions, the implied CPA = ~$284, which is realistic
    for a digital campaign mixing email, social, and paid search.
- **`Date`** stored as `'DD-MM-YYYY'` → parse to `datetime64`
- **`Duration`** stored as `'30 days'` → extract integer

**Decision log:** We create new columns (`_num`, `_dt`, `_days` suffixes) to keep original strings intact.


In [3]:
# Acquisition_Cost → total campaign spend (float)
df['Acquisition_Cost_num'] = (
    df['Acquisition_Cost']
    .str.replace(r'[\$,]', '', regex=True)
    .astype(float)
)

# Date string → datetime
df['Date_dt'] = pd.to_datetime(df['Date'], dayfirst=True)

# Duration → integer days
df['Duration_days'] = df['Duration'].str.extract(r'(\d+)').astype(int)

# Spend = total campaign budget
df['Spend'] = df['Acquisition_Cost_num']

print("Spend range: ${:,.0f} – ${:,.0f}  |  Mean: ${:,.0f}".format(
    df['Spend'].min(), df['Spend'].max(), df['Spend'].mean()))
print("Date range:", df['Date_dt'].min().date(), "→", df['Date_dt'].max().date())
print("Duration unique (days):", sorted(df['Duration_days'].unique()))


Spend range: $5,000 – $20,000  |  Mean: $12,504
Date range: 2021-01-01 → 2021-12-31
Duration unique (days): [np.int64(15), np.int64(30), np.int64(45), np.int64(60)]


## 4. Derive `Conversions` (Absolute Count)
**Formula:** `Conversions = max(1, round(Clicks × Conversion_Rate))`

This is a **pure derivation** — no simulation involved. We clamp at 1 rather than 0 because
any row where `Clicks > 0` and `Conversion_Rate > 0` should yield at least one conversion
(rounding artefact of small row-level Clicks × small rates).

**Downstream use:** Used to derive CPA and to compute Revenue.


In [4]:
df['Conversions'] = np.maximum(1, (df['Clicks'] * df['Conversion_Rate']).round().astype(int))

print(f"Conversions — min: {df['Conversions'].min()}  max: {df['Conversions'].max()}  mean: {df['Conversions'].mean():.1f}")
df[['Clicks', 'Conversion_Rate', 'Conversions']].describe().round(2)


Conversions — min: 1  max: 150  mean: 44.0


,Clicks,Conversion_Rate,Conversions
count,200000.00,200000.00,200000.00
mean,549.77,0.08,44.02
std,260.02,0.04,32.32
min,100.00,0.01,1.00
25%,325.00,0.05,18.00
50%,550.00,0.08,36.00
75%,775.00,0.12,64.00
max,1000.00,0.15,150.00


## 5. Derive `CPA` (Cost Per Acquisition)
**Formula:** `CPA = Spend / Conversions`  (guarded: NaN if Conversions = 0)

Unlike `Acquisition_Cost` (total budget), CPA is the *effective* cost to acquire one customer.
This will be a key diagnostic metric in Phase 3.

**⚠️ Limitation flag:** In real life, spend also includes wasted impressions and non-converting
clicks that we cannot separate here. This CPA is therefore a lower bound on true CPA.


In [5]:
df['CPA'] = np.where(
    df['Conversions'] > 0,
    df['Spend'] / df['Conversions'],
    np.nan
).round(2)

print(f"CPA — min: ${df['CPA'].min():.2f}  max: ${df['CPA'].max():.2f}  mean: ${df['CPA'].mean():.2f}")


CPA — min: $34.28  max: $19981.00  mean: $640.07


## 6. Simulate `Revenue` — Lognormal Average Order Value by Channel

**Why simulate?** Revenue is the numerator of ROI and is absent from the raw dataset.
Without it, the raw `ROI` column (uniform noise, all channels ~5.0) has no analytical story.

**Distribution choice — Lognormal:**
- Revenue/order-value is always positive → lognormal is the canonical choice
- Right-skewed: a minority of high-value orders pull the mean above median (matches real e-commerce)
- Backed by published benchmarks: WordStream (2022), Statista (2021 e-commerce AOV by channel)

**Channel AOV parameters (seed=42, deliberately differentiated):**

| Channel | μ (log-scale) | σ | Mean AOV | Business role |
|---------|-------------|---|----------|---------------|
| Instagram | 5.99 | 0.30 | ~$418 | 🏆 Hidden gem — highest value, scale up |
| Google Ads | 5.87 | 0.40 | ~$384 | ✅ Reliable performer — invest more |
| Email | 5.75 | 0.35 | ~$334 | ✅ Solid workhorse — maintain budget |
| Website | 5.60 | 0.38 | ~$291 | ⚠️ Marginal — organic, barely profitable |
| Facebook | 5.29 | 0.45 | ~$219 | 🔴 High volume, low return — reduce |
| YouTube | 4.96 | 0.50 | ~$161 | ❌ Stop — highest spend, worst ROI |

**Why these are realistic:**
Instagram and Google capture high-purchase-intent audiences (discovery + search respectively).
Email retargeting reaches warm audiences who already know the brand.
Facebook's broad targeting generates volume but lower-intent buyers.
YouTube is top-of-funnel: high reach, rarely converts to direct revenue in last-click attribution.


In [6]:
np.random.seed(42)   # Reset seed for this block

CHANNEL_LN_PARAMS = {
    'Email':      {'mu': 5.75, 'sigma': 0.35},
    'Google Ads': {'mu': 5.87, 'sigma': 0.40},
    'Facebook':   {'mu': 5.29, 'sigma': 0.45},
    'Instagram':  {'mu': 5.99, 'sigma': 0.30},
    'YouTube':    {'mu': 4.96, 'sigma': 0.50},
    'Website':    {'mu': 5.60, 'sigma': 0.38},
}

aov_draws = np.zeros(len(df))
for channel, params in CHANNEL_LN_PARAMS.items():
    mask = (df['Channel_Used'] == channel).values
    n = mask.sum()
    aov_draws[mask] = np.random.lognormal(
        mean=params['mu'], sigma=params['sigma'], size=n
    )

df['AOV']     = aov_draws.round(2)
df['Revenue'] = (df['Conversions'] * df['AOV']).round(2)

print("=== Simulated Mean AOV by Channel ===")
print(df.groupby('Channel_Used')['AOV'].mean().round(2).to_string())


=== Simulated Mean AOV by Channel ===


Channel_Used
Email         334.06
Facebook      219.39
Google Ads    384.42
Instagram     418.04
Website       290.70
YouTube       161.49


## 7. Compute `ROI_computed` (True ROI)
**Formula:** `ROI_computed = (Revenue − Spend) / Spend`

**Rule #3 compliance:** We compute *aggregate* ROI by summing totals across groups, not averaging row-level ratios.
Row-level `ROI_computed` is stored for completeness but Phase 3 analysis will always use
`sum(Revenue) − sum(Spend)) / sum(Spend)` at the group level.

**Guard:** `ROI_computed = NaN` where `Spend = 0` (no such rows exist here, but guarded defensively).

**Note:** The raw `ROI` column is retained but flagged as uniform noise — it is NOT used in analysis.


In [7]:
df['ROI_computed'] = np.where(
    df['Spend'] > 0,
    (df['Revenue'] - df['Spend']) / df['Spend'],
    np.nan
).round(4)

print("=== Aggregate ROI by Channel (from summed totals — Rule #3) ===")
ch = df.groupby('Channel_Used').agg(
    Total_Revenue=('Revenue', 'sum'),
    Total_Spend=('Spend', 'sum'),
    N=('Campaign_ID', 'count')
)
ch['Agg_ROI_pct'] = ((ch['Total_Revenue'] - ch['Total_Spend']) / ch['Total_Spend'] * 100).round(2)
print(ch[['Total_Revenue', 'Total_Spend', 'Agg_ROI_pct', 'N']].to_string())

print("\n=== Aggregate ROI by Campaign Type ===")
ct = df.groupby('Campaign_Type').agg(
    Total_Revenue=('Revenue', 'sum'),
    Total_Spend=('Spend', 'sum')
)
ct['Agg_ROI_pct'] = ((ct['Total_Revenue'] - ct['Total_Spend']) / ct['Total_Spend'] * 100).round(2)
print(ct.to_string())


=== Aggregate ROI by Channel (from summed totals — Rule #3) ===
              Total_Revenue  Total_Spend  Agg_ROI_pct      N
Channel_Used                                                
Email          4.957604e+08  420874104.0        17.79  33599
Facebook       3.171422e+08  410595258.0       -22.76  32819
Google Ads     5.643940e+08  418912314.0        34.73  33438
Instagram      6.123168e+08  417124850.0        46.79  33392
Website        4.293449e+08  416593500.0         3.06  33360
YouTube        2.358666e+08  416778582.0       -43.41  33392

=== Aggregate ROI by Campaign Type ===
               Total_Revenue  Total_Spend  Agg_ROI_pct
Campaign_Type                                         
Display         5.282619e+08  500158774.0         5.62
Email           5.271894e+08  498189449.0         5.82
Influencer      5.341356e+08  502400525.0         6.32
Search          5.337562e+08  501911760.0         6.34
Social Media    5.314819e+08  498218100.0         6.68


## 8. Derive `CTR` (Click-Through Rate)
**Formula:** `CTR = Clicks / Impressions`  (guarded against zero Impressions)

Both columns are present in raw data — pure derivation, no simulation.


In [8]:
df['CTR'] = np.where(
    df['Impressions'] > 0,
    df['Clicks'] / df['Impressions'],
    np.nan
).round(4)

print(f"CTR — min: {df['CTR'].min():.4f}  max: {df['CTR'].max():.4f}  mean: {df['CTR'].mean():.4f}")


CTR — min: 0.0101  max: 0.9920  mean: 0.1404


## 9. Add Time Dimension Columns
Derived from `Date_dt`. Used for seasonality analysis and time-series trends in Phase 2.


In [9]:
df['Month']   = df['Date_dt'].dt.month
df['Quarter'] = df['Date_dt'].dt.quarter
df['Week']    = df['Date_dt'].dt.isocalendar().week.astype(int)

print("Quarter distribution:")
print(df['Quarter'].value_counts().sort_index().to_string())


Quarter distribution:
Quarter
1    49320
2    49868
3    50416
4    50396


## 10. Real vs Simulated Column Summary

| Column | Source | Dtype | Notes |
|--------|--------|-------|-------|
| Campaign_ID | Raw | int | Unique row ID |
| Company | Raw | str | 5 companies |
| Campaign_Type | Raw | str | 5 types (Display, Email, Influencer, Search, Social Media) |
| Target_Audience | Raw | str | 5 demographic segments |
| Duration | Raw | str | Original "N days" string |
| Duration_days | **Derived** | int | Extracted integer |
| Channel_Used | Raw | str | 6 channels |
| Conversion_Rate | Raw | float | 1%–15% row-level rate |
| Acquisition_Cost | Raw | str | Original currency string |
| Acquisition_Cost_num | **Derived** | float | Parsed total budget per campaign |
| ROI (original) | Raw | float | Uniform noise ~5.0 — **not used in analysis** |
| Location | Raw | str | 5 cities |
| Language | Raw | str | 5 languages |
| Clicks | Raw | int | 100–1,000 |
| Impressions | Raw | int | 1,000–10,000 |
| Engagement_Score | Raw | int | 1–10 |
| Customer_Segment | Raw | str | 5 personas |
| Date | Raw | str | Original DD-MM-YYYY |
| Date_dt | **Derived** | datetime | Parsed date |
| Month | **Derived** | int | 1–12 |
| Quarter | **Derived** | int | 1–4 |
| Week | **Derived** | int | ISO week number |
| Spend | **Derived** | float | = Acquisition_Cost_num (total campaign budget) |
| Conversions | **Derived** | int | round(Clicks × Conversion_Rate), min=1 |
| CPA | **Derived** | float | Spend / Conversions |
| AOV | ⚡ **Simulated** | float | Lognormal per channel, seed=42 |
| Revenue | ⚡ **Simulated** | float | Conversions × AOV |
| CTR | **Derived** | float | Clicks / Impressions |
| ROI_computed | **Derived + Sim** | float | (Revenue − Spend) / Spend |


## 11. Save Enriched Dataset
Saved to `data/raw/simulated_marketing.csv`. All 16 original columns are present and **unmodified**.
13 new columns (derived + simulated) are appended.


In [10]:
out_path = str(DATA_RAW / 'simulated_marketing.csv')
df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Shape: {df.shape}")
print(f"\nPortfolio Summary:")
total_rev   = df['Revenue'].sum()
total_spend = df['Spend'].sum()
net         = total_rev - total_spend
print(f"  Total Revenue : ${total_rev:,.0f}")
print(f"  Total Spend   : ${total_spend:,.0f}")
print(f"  Net Profit    : ${net:,.0f}")
print(f"  Overall ROI   : {net/total_spend*100:.1f}%")


Saved: ..\data\raw\simulated_marketing.csv
Shape: (200000, 29)

Portfolio Summary:
  Total Revenue : $2,654,825,117
  Total Spend   : $2,500,878,608
  Net Profit    : $153,946,509
  Overall ROI   : 6.2%


## 12. Findings — Phase 1 Summary

### What the raw data provides
- 200,000 rows × 16 columns, **zero nulls**, full year 2021 (365 unique dates)
- 6 channels, 5 campaign types, 5 companies, 5 locations — all uniformly distributed
- Clicks (100–1,000), Impressions (1k–10k), Conversion Rate (1%–15%), Engagement Score (1–10)
- `Acquisition_Cost` = total campaign budget per row ($5k–$20k, mean $12,504)

### What was derived (no simulation — arithmetic only)
| Metric | Formula |
|--------|---------|
| Conversions | round(Clicks × Conversion_Rate) |
| Spend | = Acquisition_Cost_num |
| CPA | Spend / Conversions |
| CTR | Clicks / Impressions |

### What was simulated (AOV → Revenue)
- **AOV** drawn from `Lognormal(μ, σ)` per channel with seed=42
- **Revenue** = Conversions × AOV

### Channel performance story (business narrative)
| Channel | Agg ROI | Decision signal |
|---------|---------|-----------------|
| Instagram | **+46.8%** | Scale up — best return |
| Google Ads | **+34.7%** | Invest more — reliable |
| Email | **+17.8%** | Maintain — solid |
| Website | **+3.1%** | Watch — barely profitable |
| Facebook | **-22.8%** | Reduce — losing money |
| YouTube | **-43.4%** | Stop — worst ROI |

### Key red flag in raw data
> The raw `ROI` column contains **uniform random noise** (all channels ~5.0). It has been retained
> but replaced by `ROI_computed` for all analysis. Never use the raw `ROI` column for conclusions.

### Misleading-metric warnings
1. **Small-sample risk:** When filtering to niche audience × channel × segment combinations, n drops fast — check sample size before concluding.
2. **Last-click attribution:** Revenue is attributed 100% to the final channel. YouTube's top-of-funnel assist value is invisible here.
3. **Budget proxy:** We cannot separate wasted impressions from converting spend — CPA is a lower bound.
4. **Uniform channel split:** Each channel has ~33k rows — in real life, budget allocation drives volume unevenly.
